In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Define the CNN architecture
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # Input: 3 channels (RGB)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),  # Adjust based on input size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Define transformations for data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize the model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN(num_classes=10).to(device)  # CIFAR-10 has 10 classes
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

# Evaluate the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the test images: {100 * correct / total:.2f}%')bj

100%|██████████| 170M/170M [00:01<00:00, 101MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Epoch [1/10], Loss: 1.7865
Epoch [2/10], Loss: 1.5431
Epoch [3/10], Loss: 1.4298
Epoch [4/10], Loss: 1.3587
Epoch [5/10], Loss: 1.3075
Epoch [6/10], Loss: 1.2591
Epoch [7/10], Loss: 1.2301
Epoch [8/10], Loss: 1.2103
Epoch [9/10], Loss: 1.1913
Epoch [10/10], Loss: 1.1730
Accuracy of the model on the test images: 72.40%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Squeeze-and-Excitation Block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, _, _ = x.size()
        y = x.mean(dim=(2, 3))
        y = self.fc1(y).relu()
        y = self.fc2(y)
        return x * self.sigmoid(y).view(batch_size, channels, 1, 1).expand_as(x)

# Custom CNN with Attention
class CustomCNNWithAttention(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNNWithAttention, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            SEBlock(32), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            SEBlock(64), nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            SEBlock(128), nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            SEBlock(256)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(), nn.Linear(256 * 2 * 2, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x))

# Data augmentation and normalization
transform = transforms.Compose([
    transforms.RandomResizedCrop(32), transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
train_loader = DataLoader(datasets.CIFAR10('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)
test_loader = DataLoader(datasets.CIFAR10('./data', train=False, download=True, transform=transform), batch_size=64, shuffle=False)

# Initialize model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNNWithAttention(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(10):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch+1}/10], Loss: {loss.item():.4f}')

# Evaluation
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')

Files already downloaded and verified
Files already downloaded and verified
Epoch [1/10], Loss: 1.5075
Epoch [2/10], Loss: 1.6552
Epoch [3/10], Loss: 1.5223
Epoch [4/10], Loss: 1.1886
Epoch [5/10], Loss: 1.1247
Epoch [6/10], Loss: 1.0997
Epoch [7/10], Loss: 1.3710
Epoch [8/10], Loss: 1.0695
Epoch [9/10], Loss: 1.0359
Epoch [10/10], Loss: 0.8133
Accuracy: 61.01%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights  # Updated import for weights

# Squeeze-and-Excitation Block
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super(SEBlock, self).__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels = x.size()
        y = x.mean(dim=0)  # Global average pooling
        y = self.fc1(y).relu()
        y = self.fc2(y)
        return x * self.sigmoid(y).view(1, channels).expand_as(x)

# Custom CNN with Attention
class CustomCNNWithAttention(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNNWithAttention, self).__init__()
        self.base_model = resnet18(weights=ResNet18_Weights.DEFAULT)  # Load pre-trained ResNet
        self.base_model.fc = nn.Identity()  # Remove the final layer
        self.se_block = SEBlock(512)  # ResNet-18 outputs 512 features

        self.fc = nn.Linear(512, num_classes)  # New final layer

    def forward(self, x):
        x = self.base_model(x)  # Get features
        x = self.se_block(x)  # Apply SE Block
        x = self.fc(x)  # Classify
        return x

# Data augmentation and normalization
transform = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
train_loader = DataLoader(datasets.CIFAR10('./data', train=True, download=True, transform=transform), batch_size=64, shuffle=True)
test_loader = DataLoader(datasets.CIFAR10('./data', train=False, download=True, transform=transform), batch_size=64, shuffle=False)

# Initialize model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNNWithAttention(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop with Early Stopping
best_accuracy = 0
patience = 5
patience_counter = 0

for epoch in range(50):  # Increase epochs for better training
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'Epoch [{epoch+1}/50], Loss: {running_loss/len(train_loader):.4f}')

    # Evaluation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy: {accuracy:.2f}%')

    # Early stopping
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

Files already downloaded and verified
Files already downloaded and verified
Epoch [1/50], Loss: 1.4201
Accuracy: 57.59%
Epoch [2/50], Loss: 1.1938
Accuracy: 58.61%
Epoch [3/50], Loss: 1.1006
Accuracy: 62.92%
Epoch [4/50], Loss: 1.0492
Accuracy: 62.30%
Epoch [5/50], Loss: 1.0136
Accuracy: 65.44%
Epoch [6/50], Loss: 0.9797
Accuracy: 65.87%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

# Define the CNN architecture
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),  # Adjust based on input size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Define transformations for data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize the model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN(num_classes=10).to(device)  # CIFAR-10 has 10 classes
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

# Early stopping parameters
best_loss = float('inf')
patience = 5
patience_counter = 0

# Training loop
num_epochs = 20  # Increase the number of epochs
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()  # Step the learning rate scheduler

    # Evaluate the model on the training set
    model.eval()
    train_loss = running_loss / len(train_loader)

    # Check validation loss for early stopping
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(test_loader)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Check for early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        best_model = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

# Evaluate the model on the test set
model.load_state_dict(best_model)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the test images: {100 * correct / total:.2f}%')

100%|██████████| 170M/170M [00:04<00:00, 38.4MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Epoch [1/20], Train Loss: 1.7784, Val Loss: 1.3390
Epoch [2/20], Train Loss: 1.5105, Val Loss: 1.1524
Epoch [3/20], Train Loss: 1.3956, Val Loss: 1.0571
Epoch [4/20], Train Loss: 1.3121, Val Loss: 0.9817
Epoch [5/20], Train Loss: 1.2668, Val Loss: 0.9040
Epoch [6/20], Train Loss: 1.1551, Val Loss: 0.8030
Epoch [7/20], Train Loss: 1.1117, Val Loss: 0.7740
Epoch [8/20], Train Loss: 1.0968, Val Loss: 0.7665
Epoch [9/20], Train Loss: 1.0751, Val Loss: 0.7633
Epoch [10/20], Train Loss: 1.0710, Val Loss: 0.7414
Epoch [11/20], Train Loss: 1.0533, Val Loss: 0.7361
Epoch [12/20], Train Loss: 1.0476, Val Loss: 0.7413
Epoch [13/20], Train Loss: 1.0517, Val Loss: 0.7318
Epoch [14/20], Train Loss: 1.0500, Val Loss: 0.7336
Epoch [15/20], Train Loss: 1.0433, Val Loss: 0.7429
Epoch [16/20], Train Loss: 1.0460, Val Loss: 0.7319
Epoch [17/20], Train Loss: 1.0362, Val Loss: 0.7304
Epoch [18/20], Train Loss: 1.0417, V

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

# Define the CNN architecture with more complexity
class EnhancedCNN(nn.Module):
    def __init__(self, num_classes):
        super(EnhancedCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(512),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 1 * 1, 512),  # Adjust based on input size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Define transformations for data augmentation
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize the model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EnhancedCNN(num_classes=10).to(device)  # CIFAR-10 has 10 classes
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)  # Adding weight decay

# Learning rate scheduler
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

# Early stopping parameters
best_loss = float('inf')
patience = 5
patience_counter = 0

# Training loop
num_epochs = 60  # Increase the number of epochs
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()  # Step the learning rate scheduler

    # Evaluate the model on the training set
    model.eval()
    train_loss = running_loss / len(train_loader)

    # Check validation loss for early stopping
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(test_loader)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Check for early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        best_model = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

# Evaluate the model on the test set
model.load_state_dict(best_model)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the test images: {100 * correct / total:.2f}%')

Files already downloaded and verified
Files already downloaded and verified
Epoch [1/60], Train Loss: 1.5223, Val Loss: 1.1084
Epoch [2/60], Train Loss: 1.1399, Val Loss: 0.9028
Epoch [3/60], Train Loss: 0.9819, Val Loss: 0.8413
Epoch [4/60], Train Loss: 0.8905, Val Loss: 0.7424
Epoch [5/60], Train Loss: 0.8239, Val Loss: 0.6812
Epoch [6/60], Train Loss: 0.7049, Val Loss: 0.5978
Epoch [7/60], Train Loss: 0.6678, Val Loss: 0.5793
Epoch [8/60], Train Loss: 0.6536, Val Loss: 0.5691
Epoch [9/60], Train Loss: 0.6365, Val Loss: 0.5591
Epoch [10/60], Train Loss: 0.6253, Val Loss: 0.5529
Epoch [11/60], Train Loss: 0.6066, Val Loss: 0.5517
Epoch [12/60], Train Loss: 0.6062, Val Loss: 0.5511
Epoch [13/60], Train Loss: 0.6067, Val Loss: 0.5507
Epoch [14/60], Train Loss: 0.5974, Val Loss: 0.5501
Epoch [15/60], Train Loss: 0.5990, Val Loss: 0.5501
Epoch [16/60], Train Loss: 0.5974, Val Loss: 0.5492
Epoch [17/60], Train Loss: 0.5974, Val Loss: 0.5484
Epoch [18/60], Train Loss: 0.5965, Val Loss: 0.54

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import StepLR

# Define the CNN architecture
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 2 * 2, 512),  # Adjust based on input size
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Define transformations for data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(32),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Initialize the model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomCNN(num_classes=10).to(device)  # CIFAR-10 has 10 classes
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = StepLR(optimizer, step_size=5, gamma=0.1)

# Early stopping parameters
best_loss = float('inf')
patience = 5
patience_counter = 0

# Training loop
num_epochs = 40
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()  # Step the learning rate scheduler

    # Evaluate the model on the training set
    model.eval()
    train_loss = running_loss / len(train_loader)

    # Check validation loss for early stopping
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

    val_loss /= len(test_loader)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

    # Check for early stopping
    if val_loss < best_loss:
        best_loss = val_loss
        best_model = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

# Evaluate the model on the test set
model.load_state_dict(best_model)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the test images: {110 * correct / total:.2f}%')

Files already downloaded and verified
Files already downloaded and verified
Epoch [1/40], Train Loss: 1.7708, Val Loss: 1.3094
Epoch [2/40], Train Loss: 1.5120, Val Loss: 1.1377
Epoch [3/40], Train Loss: 1.3914, Val Loss: 0.9649
Epoch [4/40], Train Loss: 1.3180, Val Loss: 0.9450
Epoch [5/40], Train Loss: 1.2645, Val Loss: 0.9559
Epoch [6/40], Train Loss: 1.1577, Val Loss: 0.7985
Epoch [7/40], Train Loss: 1.1140, Val Loss: 0.7763
Epoch [8/40], Train Loss: 1.0865, Val Loss: 0.7663
Epoch [9/40], Train Loss: 1.0745, Val Loss: 0.7416
Epoch [10/40], Train Loss: 1.0563, Val Loss: 0.7375
Epoch [11/40], Train Loss: 1.0475, Val Loss: 0.7316
Epoch [12/40], Train Loss: 1.0498, Val Loss: 0.7308
Epoch [13/40], Train Loss: 1.0496, Val Loss: 0.7311
Epoch [14/40], Train Loss: 1.0388, Val Loss: 0.7325
Epoch [15/40], Train Loss: 1.0369, Val Loss: 0.7264
Epoch [16/40], Train Loss: 1.0418, Val Loss: 0.7173
Epoch [17/40], Train Loss: 1.0341, Val Loss: 0.7267
Epoch [18/40], Train Loss: 1.0414, Val Loss: 0.72